<a href="https://colab.research.google.com/github/angelc137/colab/blob/main/TareaRedes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Instalar mlflow y la librería de dagshub
!pip install -q mlflow dagshub

# 2. Inicializar la conexión con DagsHub
import dagshub
import mlflow

# Reemplaza con tu usuario de DagsHub y el nombre que le diste al repositorio
DAGSHUB_USERNAME = "angelc137"
REPO_NAME = "colab"

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=REPO_NAME, mlflow=True)

In [ ]:
# 1. Instalar la librería de Kaggle (suele venir preinstalada, pero aseguramos)
!pip install -q kaggle

# 2. Configurar tus credenciales de Kaggle
# Reemplaza con los valores de tu archivo kaggle.json
import os
os.environ['KAGGLE_USERNAME'] = "angelchavarria137"
os.environ['KAGGLE_KEY'] = "1f2a410e11aa377b2a24177b8ba9458c"
# 3. Descargar el dataset de fraude con tarjeta de crédito
# El identificador en Kaggle es 'mlg-ulb/creditcardfraud'
!kaggle datasets download -d mlg-ulb/creditcardfraud

# 4. Descomprimir el archivo zip que se descarga
!unzip creditcardfraud.zip

In [ ]:
import pandas as pd

# Cargar el dataset directamente desde el almacenamiento local de Colab
df = pd.read_csv('creditcard.csv')
print(df.shape)
df.head()

In [ ]:
# Contar cuántas transacciones son legítimas (0) y cuántas son fraude (1)
print(df['Class'].value_counts())

# Verlo en porcentaje
print("\nPorcentaje de distribución:")
print(df['Class'].value_counts(normalize=True) * 100)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. Limpieza estricta: Escalar montos y eliminar la columna Time (que mete mucho ruido)
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))

# Tiramos 'Time' por completo para evitar que confunda a la red con valores secuenciales gigantes
if 'Time' in df.columns:
    df = df.drop(['Time'], axis=1)

# 2. Separar características y etiquetas
X = df.drop('Class', axis=1).values # .values lo convierte a arreglos limpios de NumPy
y = df['Class'].values

# 3. División estratificada
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Aplicar SMOTE sobre los arreglos ya escalados
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("¡Datos perfectamente escalados y balanceados!")

In [ ]:
from imblearn.over_sampling import SMOTE

# Aplicamos SMOTE únicamente al set de entrenamiento
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Distribución original en entrenamiento:", np.bincount(y_train))
print("Distribución balanceada con SMOTE:", np.bincount(y_train_res))

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import layers, models, callbacks

import mlflow.keras

mlflow.keras.autolog()

model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

# 2. Compilar el modelo
# Usamos métricas clave: Precisión (evitar falsos positivos) y Recall (detectar el máximo de fraudes posible)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)
with mlflow.start_run(run_name="Experimento_Red_Densa_SMOTE"):

    # Entrenar el modelo de forma normal
    history = model.fit(
        X_train_res, y_train_res,
        epochs=15,
        batch_size=2048,
        validation_data=(X_test, y_test),
        verbose=1
    )
model.summary()

In [33]:
# Cargar la extensión de TensorBoard
%load_ext tensorboard

# Abrir el panel de control interactivo
%tensorboard --logdir logs

Error: No se encontró el archivo 'TareaRedes.ipynb' en la carpeta de Colab. Por favor verifica el nombre en la barra lateral izquierda.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Hacer las predicciones con el conjunto de prueba real
y_pred_prob = model.predict(X_test)
# Como la salida es una probabilidad (sigmoid), si es >= 0.5 lo clasificamos como fraude (1)
y_pred = (y_pred_prob >= 0.5).astype(int)

# 2. Generar el reporte de clasificación (Precision, Recall, F1-Score)
print("--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_test, y_pred, target_names=['Legítima', 'Fraude']))

# 3. Calcular la Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)

# 4. Graficar la Matriz de Confusión de forma elegante
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicho Legítima', 'Predicho Fraude'],
            yticklabels=['Real Legítima', 'Real Fraude'])
plt.title('Matriz de Confusión')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.show()